# Aligning heart ST data from ISS (squidpy + moscot)

Serial sections of 6.5 PCW human heart from the Human Cell Atlas https://doi.org/10.1016/j.cell.2019.11.025

This notebook uses `squidpy` with the `moscot` backend for point-to-point alignment via optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

## Load source data

In [ ]:
# Single cell data 1
fname = '../heart_data/CN73_E1.csv.gz'
df1 = pd.read_csv(fname)
print(df1.head())

In [ ]:
# Create AnnData for source
coords_source = np.column_stack([df1['x'].values, df1['y'].values])
adata_source = ad.AnnData(
    X=np.zeros((len(coords_source), 1)),
    obs=df1,
)
adata_source.obsm['spatial'] = coords_source
print(f"Source: {adata_source.n_obs} cells")

In [ ]:
# Plot source
fig, ax = plt.subplots()
ax.scatter(adata_source.obsm['spatial'][:, 0],
           adata_source.obsm['spatial'][:, 1],
           s=1, alpha=0.2)
ax.set_title('Source')
ax.set_aspect('equal')

## Load target data

In [ ]:
# Single cell data 2
fname = '../heart_data/CN73_E2.csv.gz'
df2 = pd.read_csv(fname, skiprows=[1])
print(df2.head())

In [ ]:
# Create AnnData for target
coords_target = np.column_stack([df2['x'].values, df2['y'].values])
adata_target = ad.AnnData(
    X=np.zeros((len(coords_target), 1)),
    obs=df2,
)
adata_target.obsm['spatial'] = coords_target
print(f"Target: {adata_target.n_obs} cells")

In [ ]:
# Plot both before alignment
fig, ax = plt.subplots()
ax.scatter(adata_source.obsm['spatial'][:, 0],
           adata_source.obsm['spatial'][:, 1],
           s=1, alpha=0.1, label='source')
ax.scatter(adata_target.obsm['spatial'][:, 0],
           adata_target.obsm['spatial'][:, 1],
           s=1, alpha=0.2, label='target')
ax.legend(markerscale=10)
ax.set_aspect('equal')

## Align using moscot (optimal transport)

`sq.experimental.tl.align` automatically selects the moscot backend for point-to-point alignment.

In [ ]:
# Align source to target using moscot optimal transport
sq.experimental.tl.align(
    adata_source,
    adata_target,
    method='optimal_transport',
    verbose=True,
)

## Visualize results

In [ ]:
# Plot results
aligned = adata_source.obsm['spatial_aligned']

fig, ax = plt.subplots()
ax.scatter(adata_source.obsm['spatial'][:, 0],
           adata_source.obsm['spatial'][:, 1],
           s=1, alpha=0.1, label='source')
ax.scatter(aligned[:, 0], aligned[:, 1],
           s=1, alpha=0.2, label='source aligned')
ax.scatter(adata_target.obsm['spatial'][:, 0],
           adata_target.obsm['spatial'][:, 1],
           s=1, alpha=0.2, label='target')

lgnd = plt.legend(loc='upper right', scatterpoints=1, fontsize=10)
for handle in lgnd.legend_handles:
    handle.set_sizes([10.0])

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

axes[0].scatter(adata_source.obsm['spatial'][:, 0],
                adata_source.obsm['spatial'][:, 1],
                s=1, alpha=0.1, label='source')
axes[0].scatter(adata_target.obsm['spatial'][:, 0],
                adata_target.obsm['spatial'][:, 1],
                s=1, alpha=0.1, label='target')
axes[0].set_title('Before alignment')
axes[0].legend(markerscale=10)
axes[0].set_aspect('equal')

axes[1].scatter(aligned[:, 0], aligned[:, 1],
                s=1, alpha=0.1, label='source aligned')
axes[1].scatter(adata_target.obsm['spatial'][:, 0],
                adata_target.obsm['spatial'][:, 1],
                s=1, alpha=0.1, label='target')
axes[1].set_title('After alignment')
axes[1].legend(markerscale=10)
axes[1].set_aspect('equal')

In [ ]:
print(adata_source.uns['spatial_alignment'])